# 0825_peace_011_type_expert_fold_ensemble_time_weight


## 1. 설정, 경로 탐색과 실행 로그

In [1]:
import gc
import hashlib
import json
import logging
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

EXPERIMENT_ID = "0825_peace_011_type_expert_fold_ensemble_time_weight"
RANDOM_STATE = 42
TARGET = "class"
TIME_COLUMN = "timestamp"
TYPE_COLUMN = "inspection_type"
RECORD_ID = "record_id"
DECISION_THRESHOLD = 0.5
MIN_RECALL = 0.99
TRAIN_END_FRACTION = 0.70
VALIDATION_END_FRACTION = 0.80
TIME_WEIGHT_MIN = 1.0
TIME_WEIGHT_MAX = 2.0

XGB_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_estimators": 400,
    "learning_rate": 0.05,
    "max_depth": 5,
    "min_child_weight": 10,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 5.0,
    "max_delta_step": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0,
}


def find_repo_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "notebooks").is_dir():
            return candidate.resolve()
    raise FileNotFoundError("AGENTS.md가 있는 저장소 루트를 찾지 못했습니다.")


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def find_data_pair(repo_root: Path) -> tuple[Path, Path]:
    candidates = [
        repo_root / "data" / "raw",
        repo_root.parent,
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd().parent.parent,
    ]
    checked = set()
    for directory in candidates:
        resolved = directory.resolve()
        if resolved in checked:
            continue
        checked.add(resolved)
        data_path = resolved / "dataset.csv"
        mapping_path = resolved / "mapping.json"
        if data_path.exists() and mapping_path.exists():
            return data_path, mapping_path
    raise FileNotFoundError("dataset.csv와 mapping.json 쌍을 찾지 못했습니다.")


def make_time_sample_weight(frame: pd.DataFrame, time_column: str = TIME_COLUMN):
    if len(frame) == 0:
        raise ValueError("빈 frame에는 시간 가중치를 만들 수 없습니다.")

    timestamps = pd.to_datetime(frame[time_column], utc=True)
    timestamp_ns = timestamps.astype("int64").to_numpy(dtype=np.float64, copy=False)
    min_ns = float(timestamp_ns.min())
    max_ns = float(timestamp_ns.max())

    if max_ns == min_ns:
        weights = np.full(len(frame), TIME_WEIGHT_MIN, dtype=np.float64)
        degenerate = True
    else:
        relative_position = (timestamp_ns - min_ns) / (max_ns - min_ns)
        weights = TIME_WEIGHT_MIN + relative_position * (TIME_WEIGHT_MAX - TIME_WEIGHT_MIN)
        degenerate = False

    summary = {
        "time_weight_min": float(weights.min()),
        "time_weight_max": float(weights.max()),
        "time_weight_mean": float(weights.mean()),
        "time_weight_degenerate": degenerate,
        "train_start_time": timestamps.min(),
        "train_end_time": timestamps.max(),
    }
    return weights, summary


REPO_ROOT = find_repo_root()
DATA_PATH, MAPPING_PATH = find_data_pair(REPO_ROOT)
LOG_DIR = REPO_ROOT / "docs" / "peace"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / f"{EXPERIMENT_ID}.log"

logger = logging.getLogger(EXPERIMENT_ID)
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
file_handler.setFormatter(formatter)
stream_handler = logging.StreamHandler(sys.stdout)
stream_handler.setFormatter(formatter)
logger.addHandler(file_handler)
logger.addHandler(stream_handler)
logger.propagate = False

DATA_SHA256_BEFORE = sha256_file(DATA_PATH)
MAPPING_SHA256_BEFORE = sha256_file(MAPPING_PATH)
logger.info("experiment=%s", EXPERIMENT_ID)
logger.info(
    "random_state=%d baseline_threshold=%.2f min_recall=%.2f time_weight_range=[%.1f, %.1f]",
    RANDOM_STATE,
    DECISION_THRESHOLD,
    MIN_RECALL,
    TIME_WEIGHT_MIN,
    TIME_WEIGHT_MAX,
)
logger.info("data_file=%s sha256=%s", DATA_PATH.name, DATA_SHA256_BEFORE)
logger.info("mapping_file=%s sha256=%s", MAPPING_PATH.name, MAPPING_SHA256_BEFORE)
logger.info(
    "versions python=%s pandas=%s sklearn=%s xgboost=%s",
    sys.version.split()[0], pd.__version__, sklearn.__version__, xgboost.__version__
)
logger.info("log_file=docs/peace/%s", LOG_PATH.name)
print("log saved to:", LOG_PATH.relative_to(REPO_ROOT))


2026-08-25 13:57:47,756 | INFO | experiment=0825_peace_011_type_expert_fold_ensemble_time_weight


2026-08-25 13:57:47,757 | INFO | random_state=42 baseline_threshold=0.50 min_recall=0.99 time_weight_range=[1.0, 2.0]


2026-08-25 13:57:47,757 | INFO | data_file=dataset.csv sha256=53e8568743216d556856ed69b388f6750fbfa0b8c59ad31f970515ac9eb10e62


2026-08-25 13:57:47,757 | INFO | mapping_file=mapping.json sha256=3b20f440b6d9ed0baefa662e1a6f03688befbe0f28341a3b54655d3058c6e486


2026-08-25 13:57:47,758 | INFO | versions python=3.12.7 pandas=2.2.2 sklearn=1.5.1 xgboost=3.4.1


2026-08-25 13:57:47,758 | INFO | log_file=docs/peace/0825_peace_011_type_expert_fold_ensemble_time_weight.log


log saved to: docs/peace/0825_peace_011_type_expert_fold_ensemble_time_weight.log


## 2. 원본 데이터와 매핑 검증

In [2]:
raw_df = pd.read_csv(DATA_PATH, low_memory=False)
source_index_column = raw_df.columns[0]
if source_index_column.startswith("Unnamed:") or source_index_column == "":
    raw_df = raw_df.rename(columns={source_index_column: RECORD_ID})
elif source_index_column != RECORD_ID:
    raise ValueError(f"예상하지 못한 첫 번째 컬럼: {source_index_column}")

with MAPPING_PATH.open(encoding="utf-8") as stream:
    feature_mapping = json.load(stream)

required_columns = {RECORD_ID, TIME_COLUMN, TYPE_COLUMN, TARGET}
missing_required = required_columns - set(raw_df.columns)
assert not missing_required, f"필수 컬럼 누락: {sorted(missing_required)}"
assert len(raw_df) == 440_274
assert raw_df[RECORD_ID].is_unique
assert set(raw_df[TARGET].unique()) == {0, 1}
assert raw_df[TARGET].value_counts().to_dict() == {0: 435_652, 1: 4_622}
assert set(raw_df[TYPE_COLUMN].unique()) == {0, 1, 2, 3, 4}
assert set(feature_mapping) == {"0", "1", "2", "3", "4"}

raw_df[TIME_COLUMN] = pd.to_datetime(raw_df[TIME_COLUMN], errors="raise", utc=True)
raw_df = raw_df.sort_values([TIME_COLUMN, RECORD_ID], kind="stable").reset_index(drop=True)
inspection_columns = [column for column in raw_df.columns if column.startswith("inspection_feat")]
mapped_union = set().union(*(set(columns) for columns in feature_mapping.values()))
assert len(inspection_columns) == 70
assert len(mapped_union) == 65
assert mapped_union <= set(inspection_columns)

numeric_inputs = raw_df.select_dtypes(include=[np.number]).drop(columns=[TARGET, RECORD_ID])
assert np.isfinite(numeric_inputs.to_numpy()).all()

data_summary = pd.Series(
    {
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
        "false_call_0": int((raw_df[TARGET] == 0).sum()),
        "real_defect_1": int((raw_df[TARGET] == 1).sum()),
        "real_defect_rate_pct": raw_df[TARGET].mean() * 100,
        "inspection_types": raw_df[TYPE_COLUMN].nunique(),
        "inspection_features": len(inspection_columns),
        "mapped_feature_union": len(mapped_union),
        "timestamp_start": raw_df[TIME_COLUMN].min(),
        "timestamp_end": raw_df[TIME_COLUMN].max(),
    },
    name="raw_data",
)
display(data_summary)
logger.info(
    "data_verified rows=%d columns=%d class_0=%d class_1=%d",
    len(raw_df), raw_df.shape[1], int((raw_df[TARGET] == 0).sum()), int((raw_df[TARGET] == 1).sum())
)


rows                                       440274
columns                                        78
false_call_0                               435652
real_defect_1                                4622
real_defect_rate_pct                     1.049801
inspection_types                                5
inspection_features                            70
mapped_feature_union                           65
timestamp_start         1970-06-23 03:58:55+00:00
timestamp_end           1970-11-02 14:21:28+00:00
Name: raw_data, dtype: object

2026-08-25 13:57:53,023 | INFO | data_verified rows=440274 columns=78 class_0=435652 class_1=4622


## 3. 타입별 유효 피처

In [3]:
inspection_types = sorted(raw_df[TYPE_COLUMN].unique().tolist())
meta_columns = [column for column in raw_df.columns if column.startswith("meta_feat")]
feature_columns_by_type = {}
feature_rows = []

for inspection_type in inspection_types:
    mapped_columns = feature_mapping[str(inspection_type)]
    assert len(mapped_columns) == len(set(mapped_columns))
    assert set(mapped_columns) <= set(raw_df.columns)
    selected_columns = meta_columns + mapped_columns
    feature_columns_by_type[inspection_type] = selected_columns
    feature_rows.append(
        {
            "inspection_type": inspection_type,
            "meta_features": len(meta_columns),
            "mapped_inspection_features": len(mapped_columns),
            "total_model_features": len(selected_columns),
        }
    )

feature_summary = pd.DataFrame(feature_rows).set_index("inspection_type")
display(feature_summary)
logger.info("feature_mapping_verified=%s", feature_summary.to_dict(orient="index"))


,meta_features,mapped_inspection_features,total_model_features
inspection_type,,,
0,4,44,48
1,4,52,56
2,4,65,69
3,4,65,69
4,4,21,25


2026-08-25 13:57:53,035 | INFO | feature_mapping_verified={0: {'meta_features': 4, 'mapped_inspection_features': 44, 'total_model_features': 48}, 1: {'meta_features': 4, 'mapped_inspection_features': 52, 'total_model_features': 56}, 2: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 3: {'meta_features': 4, 'mapped_inspection_features': 65, 'total_model_features': 69}, 4: {'meta_features': 4, 'mapped_inspection_features': 21, 'total_model_features': 25}}


## 4. 동일한 시간순 Train/Validation/Test 분할

In [4]:
timestamp_group_sizes = raw_df.groupby(TIME_COLUMN, sort=True).size()
cumulative_rows = timestamp_group_sizes.cumsum().to_numpy()
timestamp_index = timestamp_group_sizes.index


def boundary_at(fraction: float):
    position = int(np.searchsorted(cumulative_rows, len(raw_df) * fraction, side="left"))
    return timestamp_index[position]


train_end_time = boundary_at(TRAIN_END_FRACTION)
validation_end_time = boundary_at(VALIDATION_END_FRACTION)
train_mask = raw_df[TIME_COLUMN] <= train_end_time
validation_mask = (
    (raw_df[TIME_COLUMN] > train_end_time)
    & (raw_df[TIME_COLUMN] <= validation_end_time)
)
test_mask = raw_df[TIME_COLUMN] > validation_end_time

train_df = raw_df.loc[train_mask]
validation_df = raw_df.loc[validation_mask]
test_df = raw_df.loc[test_mask]
assert train_df[TIME_COLUMN].max() < validation_df[TIME_COLUMN].min()
assert set(train_df[TIME_COLUMN]).isdisjoint(set(validation_df[TIME_COLUMN]))
assert validation_df[TIME_COLUMN].max() < test_df[TIME_COLUMN].min()
assert set(validation_df[TIME_COLUMN]).isdisjoint(set(test_df[TIME_COLUMN]))
assert int(train_mask.sum() + validation_mask.sum() + test_mask.sum()) == len(raw_df)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "rows": len(train_df),
            "positive_samples": int(train_df[TARGET].sum()),
            "positive_rate_pct": train_df[TARGET].mean() * 100,
            "timestamp_groups": train_df[TIME_COLUMN].nunique(),
            "start_time": train_df[TIME_COLUMN].min(),
            "end_time": train_df[TIME_COLUMN].max(),
        },
        {
            "split": "validation",
            "rows": len(validation_df),
            "positive_samples": int(validation_df[TARGET].sum()),
            "positive_rate_pct": validation_df[TARGET].mean() * 100,
            "timestamp_groups": validation_df[TIME_COLUMN].nunique(),
            "start_time": validation_df[TIME_COLUMN].min(),
            "end_time": validation_df[TIME_COLUMN].max(),
        },
        {
            "split": "test",
            "rows": len(test_df),
            "positive_samples": int(test_df[TARGET].sum()),
            "positive_rate_pct": test_df[TARGET].mean() * 100,
            "timestamp_groups": test_df[TIME_COLUMN].nunique(),
            "start_time": test_df[TIME_COLUMN].min(),
            "end_time": test_df[TIME_COLUMN].max(),
        },
    ]
).set_index("split")
display(split_summary)
evaluation_policy = pd.Series(
    {
        "model_selection_uses_test": False,
        "threshold_selected_on_test": False,
        "fixed_test_threshold": DECISION_THRESHOLD,
    },
    name="evaluation_policy",
)
display(evaluation_policy)
logger.info("split_summary=%s", split_summary.reset_index().to_dict(orient="records"))
logger.info("test_policy model_selection=False threshold=%.2f", DECISION_THRESHOLD)


,rows,positive_samples,positive_rate_pct,timestamp_groups,start_time,end_time
split,,,,,,
train,308196,1940,0.629470,29249,1970-06-23 03:58:55+00:00,1970-10-05 00:29:59+00:00
validation,44026,357,0.810884,3400,1970-10-05 00:30:30+00:00,1970-10-13 16:54:14+00:00
test,88052,2325,2.640485,7093,1970-10-13 16:54:52+00:00,1970-11-02 14:21:28+00:00


model_selection_uses_test     False
threshold_selected_on_test    False
fixed_test_threshold            0.5
Name: evaluation_policy, dtype: object

2026-08-25 13:57:53,468 | INFO | split_summary=[{'split': 'train', 'rows': 308196, 'positive_samples': 1940, 'positive_rate_pct': 0.6294695583330089, 'timestamp_groups': 29249, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-10-05 00:29:59+0000', tz='UTC')}, {'split': 'validation', 'rows': 44026, 'positive_samples': 357, 'positive_rate_pct': 0.8108844773542907, 'timestamp_groups': 3400, 'start_time': Timestamp('1970-10-05 00:30:30+0000', tz='UTC'), 'end_time': Timestamp('1970-10-13 16:54:14+0000', tz='UTC')}, {'split': 'test', 'rows': 88052, 'positive_samples': 2325, 'positive_rate_pct': 2.640485167855358, 'timestamp_groups': 7093, 'start_time': Timestamp('1970-10-13 16:54:52+0000', tz='UTC'), 'end_time': Timestamp('1970-11-02 14:21:28+0000', tz='UTC')}]


2026-08-25 13:57:53,468 | INFO | test_policy model_selection=False threshold=0.50


## 5. 동일한 평가 지표와 임계값 선택 함수

In [5]:
def evaluate_predictions(y_true, prediction, probability):
    y_true = np.asarray(y_true, dtype=np.int8)
    prediction = np.asarray(prediction, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    has_both_classes = np.unique(y_true).size == 2
    has_positive = (tp + fn) > 0
    return {
        "rows": len(y_true),
        "positive_samples": int(y_true.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "accuracy": accuracy_score(y_true, prediction),
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "false_call_reduction": tn / (tn + fp) if (tn + fp) else np.nan,
        "f1": f1_score(y_true, prediction, zero_division=0) if has_positive else np.nan,
        "roc_auc": roc_auc_score(y_true, probability) if has_both_classes else np.nan,
        "pr_auc": average_precision_score(y_true, probability) if has_both_classes else np.nan,
    }


def evaluate_probabilities(y_true, probability, threshold=DECISION_THRESHOLD):
    probability = np.asarray(probability, dtype=np.float64)
    prediction = (probability >= threshold).astype(np.int8)
    return evaluate_predictions(y_true, prediction, probability)


def select_threshold(y_true, probability, min_recall=MIN_RECALL):
    """Recall 제약을 만족하며 False Call Reduction이 최대인 threshold를 선택한다."""
    y_true = np.asarray(y_true, dtype=np.int8)
    probability = np.asarray(probability, dtype=np.float64)
    if np.unique(y_true).size != 2:
        raise ValueError("임계값 선택에는 positive와 negative가 모두 필요합니다.")

    order = np.argsort(-probability, kind="stable")
    sorted_probability = probability[order]
    sorted_target = y_true[order]
    cumulative_tp = np.cumsum(sorted_target == 1)
    cumulative_fp = np.cumsum(sorted_target == 0)
    group_ends = np.flatnonzero(
        np.r_[sorted_probability[:-1] != sorted_probability[1:], True]
    )

    thresholds = sorted_probability[group_ends]
    tp = cumulative_tp[group_ends]
    fp = cumulative_fp[group_ends]
    total_positive = int((y_true == 1).sum())
    total_negative = int((y_true == 0).sum())
    recall = tp / total_positive
    false_call_reduction = 1.0 - (fp / total_negative)
    feasible = np.flatnonzero(recall >= min_recall)
    if feasible.size == 0:
        raise RuntimeError(f"Recall {min_recall:.2%} 조건을 만족하는 threshold가 없습니다.")

    best_local = np.lexsort(
        (thresholds[feasible], recall[feasible], false_call_reduction[feasible])
    )[-1]
    best = feasible[best_local]
    selected_threshold = float(thresholds[best])
    metrics = evaluate_probabilities(y_true, probability, selected_threshold)
    return {"threshold": selected_threshold, "min_recall": min_recall, **metrics}


# 최적화 구현이 작은 합성 예제의 완전 탐색과 같은 결과인지 검증한다.
_test_y = np.array([1, 0, 1, 0, 1, 0], dtype=np.int8)
_test_probability = np.array([0.9, 0.8, 0.7, 0.6, 0.4, 0.2])
_optimized = select_threshold(_test_y, _test_probability, min_recall=2 / 3)
_reference_rows = []
for _threshold in np.unique(_test_probability):
    _metrics = evaluate_probabilities(_test_y, _test_probability, _threshold)
    if _metrics["recall"] >= 2 / 3:
        _reference_rows.append((_metrics["false_call_reduction"], _metrics["recall"], _threshold))
_reference = max(_reference_rows)
assert np.isclose(_optimized["threshold"], _reference[2])
logger.info("threshold_selector_unit_test=PASS")

def make_preprocessor(feature_columns):
    categorical = [column for column in meta_columns if column in feature_columns]
    continuous = [column for column in feature_columns if column not in categorical]
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore", dtype=np.float32),
                categorical,
            ),
            ("continuous", "passthrough", continuous),
        ],
        sparse_threshold=1.0,
        verbose_feature_names_out=True,
    )


def evaluate_calibration_and_future(calibration_frame, calibration_probability, evaluation_frame, evaluation_probability, stage_name):
    global_selection = select_threshold(calibration_frame[TARGET], calibration_probability, min_recall=MIN_RECALL)
    type_thresholds = {}
    type_prediction = pd.Series(np.nan, index=evaluation_frame.index, dtype="float64")
    threshold_rows = [{"stage": stage_name, "scope": "global", **global_selection}]
    type_rows = []
    for inspection_type in inspection_types:
        type_calibration = calibration_frame.loc[calibration_frame[TYPE_COLUMN] == inspection_type]
        selection = select_threshold(type_calibration[TARGET], calibration_probability.loc[type_calibration.index], min_recall=MIN_RECALL)
        type_thresholds[inspection_type] = selection["threshold"]
        threshold_rows.append({"stage": stage_name, "scope": f"type_{inspection_type}", **selection})
        type_evaluation = evaluation_frame.loc[evaluation_frame[TYPE_COLUMN] == inspection_type]
        type_probability = evaluation_probability.loc[type_evaluation.index]
        prediction = (type_probability >= selection["threshold"]).astype("int8")
        type_prediction.loc[type_evaluation.index] = prediction
        metrics = evaluate_predictions(type_evaluation[TARGET], prediction, type_probability)
        type_rows.append({"stage": stage_name, "inspection_type": inspection_type, "threshold": selection["threshold"], **metrics})
    strategy_metrics = {
        "fixed_0.5": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, DECISION_THRESHOLD),
        "global_threshold": evaluate_probabilities(evaluation_frame[TARGET], evaluation_probability, global_selection["threshold"]),
        "type_specific_thresholds": evaluate_predictions(evaluation_frame[TARGET], type_prediction, evaluation_probability),
    }
    metric_rows = [{"stage": stage_name, "strategy": strategy, **metrics} for strategy, metrics in strategy_metrics.items()]
    return {"global_selection": global_selection, "type_thresholds": type_thresholds, "threshold_rows": threshold_rows, "type_rows": type_rows, "metric_rows": metric_rows}

2026-08-25 13:57:53,495 | INFO | threshold_selector_unit_test=PASS


## 6. 동일한 3-Fold Expanding Walk-forward 구간

In [6]:
WALK_FORWARD_SPECS = [
    {
        "fold": "fold_1",
        "train_start": 0.00,
        "train_end": 0.30,
        "calibration_start": 0.30,
        "calibration_end": 0.40,
        "evaluation_start": 0.40,
        "evaluation_end": 0.50,
    },
    {
        "fold": "fold_2",
        "train_start": 0.00,
        "train_end": 0.40,
        "calibration_start": 0.40,
        "calibration_end": 0.50,
        "evaluation_start": 0.50,
        "evaluation_end": 0.60,
    },
    {
        "fold": "fold_3",
        "train_start": 0.00,
        "train_end": 0.50,
        "calibration_start": 0.50,
        "calibration_end": 0.60,
        "evaluation_start": 0.60,
        "evaluation_end": 0.70,
    },
]

walk_forward_boundaries = {
    fraction: boundary_at(fraction)
    for fraction in [0.30, 0.40, 0.50, 0.60, 0.70]
}
walk_forward_segments = {}
walk_forward_split_rows = []

for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    train_end = walk_forward_boundaries[spec["train_end"]]
    calibration_start = walk_forward_boundaries[spec["calibration_start"]]
    calibration_end = walk_forward_boundaries[spec["calibration_end"]]
    evaluation_start = walk_forward_boundaries[spec["evaluation_start"]]
    evaluation_end = walk_forward_boundaries[spec["evaluation_end"]]

    segments = {
        "train": raw_df.loc[raw_df[TIME_COLUMN] <= train_end],
        "calibration": raw_df.loc[
            (raw_df[TIME_COLUMN] > calibration_start)
            & (raw_df[TIME_COLUMN] <= calibration_end)
        ],
        "evaluation": raw_df.loc[
            (raw_df[TIME_COLUMN] > evaluation_start)
            & (raw_df[TIME_COLUMN] <= evaluation_end)
        ],
    }
    assert segments["train"][TIME_COLUMN].max() < segments["calibration"][TIME_COLUMN].min()
    assert segments["calibration"][TIME_COLUMN].max() < segments["evaluation"][TIME_COLUMN].min()
    assert set(segments["train"][TIME_COLUMN]).isdisjoint(segments["calibration"][TIME_COLUMN])
    assert set(segments["calibration"][TIME_COLUMN]).isdisjoint(segments["evaluation"][TIME_COLUMN])
    walk_forward_segments[fold_name] = segments

    for segment_name, frame in segments.items():
        walk_forward_split_rows.append(
            {
                "fold": fold_name,
                "segment": segment_name,
                "rows": len(frame),
                "positive_samples": int(frame[TARGET].sum()),
                "positive_rate_pct": frame[TARGET].mean() * 100,
                "timestamp_groups": frame[TIME_COLUMN].nunique(),
                "start_time": frame[TIME_COLUMN].min(),
                "end_time": frame[TIME_COLUMN].max(),
            }
        )

walk_forward_split_summary = pd.DataFrame(walk_forward_split_rows).set_index(
    ["fold", "segment"]
)
display(walk_forward_split_summary)
logger.info(
    "walk_forward_split_summary=%s",
    walk_forward_split_summary.reset_index().to_dict(orient="records"),
)


rows  positive_samples  positive_rate_pct  \
fold   segment                                                    
fold_1 train        132137              1223           0.925555   
       calibration   43979               200           0.454763   
       evaluation    44040               326           0.740236   
fold_2 train        176116              1423           0.807990   
       calibration   44040               326           0.740236   
       evaluation    44187               152           0.343993   
fold_3 train        220156              1749           0.794437   
       calibration   44187               152           0.343993   
       evaluation    43853                39           0.088933   

                    timestamp_groups                start_time  \
fold   segment                                                   
fold_1 train                   15230 1970-06-23 03:58:55+00:00   
       calibration              1251 1970-08-18 06:51:41+00:00   
       evaluation               5415 1970-08-21 23:33:55+00:00   
fold_2 train                   16481 1970-06-23 03:58:55+00:00   
       calibration              5415 1970-08-21 23:33:55+00:00   
       evaluation               4167 1970-09-15 06:47:13+00:00   
fold_3 train                   21896 1970-06-23 03:58:55+00:00   
       calibration              4167 1970-09-15 06:47:13+00:00   
       evaluation               3186 1970-09-28 05:11:13+00:00   

                                    end_time  
fold   segment                                
fold_1 train       1970-08-18 06:51:10+00:00  
       calibration 1970-08-21 23:32:59+00:00  
       evaluation  1970-09-15 06:46:33+00:00  
fold_2 train       1970-08-21 23:32:59+00:00  
       calibration 1970-09-15 06:46:33+00:00  
       evaluation  1970-09-28 05:10:37+00:00  
fold_3 train       1970-09-15 06:46:33+00:00  
       calibration 1970-09-28 05:10:37+00:00  
       evaluation  1970-10-05 00:29:59+00:00

2026-08-25 13:57:54,334 | INFO | walk_forward_split_summary=[{'fold': 'fold_1', 'segment': 'train', 'rows': 132137, 'positive_samples': 1223, 'positive_rate_pct': 0.9255545380930399, 'timestamp_groups': 15230, 'start_time': Timestamp('1970-06-23 03:58:55+0000', tz='UTC'), 'end_time': Timestamp('1970-08-18 06:51:10+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'calibration', 'rows': 43979, 'positive_samples': 200, 'positive_rate_pct': 0.4547625002842266, 'timestamp_groups': 1251, 'start_time': Timestamp('1970-08-18 06:51:41+0000', tz='UTC'), 'end_time': Timestamp('1970-08-21 23:32:59+0000', tz='UTC')}, {'fold': 'fold_1', 'segment': 'evaluation', 'rows': 44040, 'positive_samples': 326, 'positive_rate_pct': 0.740236148955495, 'timestamp_groups': 5415, 'start_time': Timestamp('1970-08-21 23:33:55+0000', tz='UTC'), 'end_time': Timestamp('1970-09-15 06:46:33+0000', tz='UTC')}, {'fold': 'fold_2', 'segment': 'train', 'rows': 176116, 'positive_samples': 1423, 'positive_rate_pct': 0.807990188

## 7. 누적 시간 체크포인트별 Fold 모델 학습 + 타입별 시간 가중치


In [7]:
ENSEMBLE_CHECKPOINTS = [0.30, 0.40, 0.50, 0.70]
FOLD_MEMBER_CHECKPOINTS = {"fold_1": [0.30], "fold_2": [0.30, 0.40], "fold_3": [0.30, 0.40, 0.50]}
FINAL_MEMBER_CHECKPOINTS = ENSEMBLE_CHECKPOINTS.copy()

prediction_targets = {}
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    prediction_targets[f"{fold_name}_calibration"] = walk_forward_segments[fold_name]["calibration"]
    prediction_targets[f"{fold_name}_evaluation"] = walk_forward_segments[fold_name]["evaluation"]
prediction_targets["final_validation"] = validation_df
prediction_targets["final_test"] = test_df

checkpoint_predictions = {
    checkpoint: {
        target_name: pd.Series(np.nan, index=frame.index, dtype="float64")
        for target_name, frame in prediction_targets.items()
        if frame[TIME_COLUMN].min() > walk_forward_boundaries[checkpoint]
    }
    for checkpoint in ENSEMBLE_CHECKPOINTS
}
ensemble_training_rows = []
for checkpoint in ENSEMBLE_CHECKPOINTS:
    checkpoint_train = raw_df.loc[raw_df[TIME_COLUMN] <= walk_forward_boundaries[checkpoint]]
    for inspection_type in inspection_types:
        feature_columns = feature_columns_by_type[inspection_type]
        type_train = checkpoint_train.loc[checkpoint_train[TYPE_COLUMN] == inspection_type]
        y_train = type_train[TARGET].astype("int8")
        assert y_train.nunique() == 2
        sample_weight, weight_summary = make_time_sample_weight(type_train)
        preprocessor = make_preprocessor(feature_columns)
        X_train = preprocessor.fit_transform(type_train[feature_columns])
        model = XGBClassifier(**XGB_PARAMS)
        model.fit(X_train, y_train, sample_weight=sample_weight, verbose=False)
        for target_name, probability_series in checkpoint_predictions[checkpoint].items():
            target_frame = prediction_targets[target_name]
            type_target = target_frame.loc[target_frame[TYPE_COLUMN] == inspection_type]
            X_target = preprocessor.transform(type_target[feature_columns])
            probability_series.loc[type_target.index] = model.predict_proba(X_target)[:, 1]
            del X_target
        ensemble_training_rows.append({
            "checkpoint": checkpoint,
            "inspection_type": inspection_type,
            "train_rows": len(type_train),
            "train_positive": int(y_train.sum()),
            "raw_features": len(feature_columns),
            "encoded_features": X_train.shape[1],
            "time_weight_min": weight_summary["time_weight_min"],
            "time_weight_max": weight_summary["time_weight_max"],
            "time_weight_mean": weight_summary["time_weight_mean"],
            "time_weight_degenerate": weight_summary["time_weight_degenerate"],
            "trees": model.get_booster().num_boosted_rounds(),
        })
        logger.info(
            "ensemble_member_fit_done checkpoint=%.2f type=%d rows=%d positive=%d weight_min=%.6f weight_max=%.6f weight_mean=%.6f weight_degenerate=%s",
            checkpoint,
            inspection_type,
            len(type_train),
            int(y_train.sum()),
            weight_summary["time_weight_min"],
            weight_summary["time_weight_max"],
            weight_summary["time_weight_mean"],
            weight_summary["time_weight_degenerate"],
        )
        del preprocessor, model, X_train, sample_weight
        gc.collect()
for checkpoint, target_map in checkpoint_predictions.items():
    for target_name, probability in target_map.items():
        assert probability.notna().all(), (checkpoint, target_name)
ensemble_training_summary = pd.DataFrame(ensemble_training_rows).set_index(["checkpoint", "inspection_type"])
display(ensemble_training_summary)
logger.info("ensemble_members_trained=%d", len(ensemble_training_rows))

2026-08-25 13:57:54,967 | INFO | ensemble_member_fit_done checkpoint=0.30 type=0 rows=28277 positive=32 weight_min=1.000000 weight_max=2.000000 weight_mean=1.516054 weight_degenerate=False


2026-08-25 13:57:55,628 | INFO | ensemble_member_fit_done checkpoint=0.30 type=1 rows=22698 positive=269 weight_min=1.000000 weight_max=2.000000 weight_mean=1.475024 weight_degenerate=False


2026-08-25 13:57:57,161 | INFO | ensemble_member_fit_done checkpoint=0.30 type=2 rows=42288 positive=408 weight_min=1.000000 weight_max=2.000000 weight_mean=1.593579 weight_degenerate=False


2026-08-25 13:57:58,311 | INFO | ensemble_member_fit_done checkpoint=0.30 type=3 rows=37264 positive=510 weight_min=1.000000 weight_max=2.000000 weight_mean=1.444547 weight_degenerate=False


2026-08-25 13:57:58,401 | INFO | ensemble_member_fit_done checkpoint=0.30 type=4 rows=1610 positive=4 weight_min=1.000000 weight_max=2.000000 weight_mean=1.563279 weight_degenerate=False


...

2026-08-25 13:58:08,470 | INFO | ensemble_member_fit_done checkpoint=0.70 type=2 rows=100470 positive=588 weight_min=1.000000 weight_max=2.000000 weight_mean=1.571788 weight_degenerate=False


2026-08-25 13:58:09,686 | INFO | ensemble_member_fit_done checkpoint=0.70 type=3 rows=100740 positive=648 weight_min=1.000000 weight_max=2.000000 weight_mean=1.580274 weight_degenerate=False


2026-08-25 13:58:09,786 | INFO | ensemble_member_fit_done checkpoint=0.70 type=4 rows=3813 positive=13 weight_min=1.000000 weight_max=2.000000 weight_mean=1.556784 weight_degenerate=False


train_rows  train_positive  raw_features  \
checkpoint inspection_type                                             
0.3        0                     28277              32            48   
           1                     22698             269            56   
           2                     42288             408            69   
           3                     37264             510            69   
           4                      1610               4            25   
0.4        0                     36685              43            48   
           1                     26566             289            56   
           2                     58736             500            69   
           3                     51683             583            69   
           4                      2446               8            25   
0.5        0                     43181              93            48   
           1                     29184             475            56   
           2                     77700             549            69   
           3                     67320             622            69   
           4                      2771              10            25   
0.7        0                     64273             111            48   
           1                     38900             580            56   
           2                    100470             588            69   
           3                    100740             648            69   
           4                      3813              13            25   

                            encoded_features  time_weight_min  \
checkpoint inspection_type                                      
0.3        0                              80              1.0   
           1                             106              1.0   
           2                             114              1.0   
           3                             107              1.0   
           4                              47              1.0   
0.4        0                              82              1.0   
           1                             110              1.0   
           2                             114              1.0   
           3                             107              1.0   
           4                              47              1.0   
0.5        0                              84              1.0   
           1                             111              1.0   
           2                             115              1.0   
           3                             108              1.0   
           4                              50              1.0   
0.7        0                              88              1.0   
           1                             113              1.0   
           2                             117              1.0   
           3                             109              1.0   
           4                              53              1.0   

                            time_weight_max  time_weight_mean  \
checkpoint inspection_type                                      
0.3        0                            2.0          1.516054   
           1                            2.0          1.475024   
           2                            2.0          1.593579   
           3                            2.0          1.444547   
           4                            2.0          1.563279   
0.4        0                            2.0          1.591712   
           1                            2.0          1.520408   
           2                            2.0          1.669343   
           3                            2.0          1.567349   
           4                            2.0          1.671896   
0.5        0                            2.0          1.495288   
           1                            2.0          1.416470   
           2                            2.0          1.578101   
           3                         

2026-08-25 13:58:09,819 | INFO | ensemble_members_trained=20


## 8. Walk-forward 미래 Evaluation 결과

In [8]:
def mean_checkpoint_probability(checkpoints, target_name):
    probabilities = [checkpoint_predictions[checkpoint][target_name].to_numpy() for checkpoint in checkpoints]
    return pd.Series(np.mean(np.vstack(probabilities), axis=0), index=prediction_targets[target_name].index, dtype="float64")

walk_threshold_rows, walk_metric_rows, walk_type_rows = [], [], []
for spec in WALK_FORWARD_SPECS:
    fold_name = spec["fold"]
    members = FOLD_MEMBER_CHECKPOINTS[fold_name]
    result = evaluate_calibration_and_future(
        walk_forward_segments[fold_name]["calibration"], mean_checkpoint_probability(members, f"{fold_name}_calibration"),
        walk_forward_segments[fold_name]["evaluation"], mean_checkpoint_probability(members, f"{fold_name}_evaluation"), fold_name,
    )
    walk_threshold_rows.extend(result["threshold_rows"])
    walk_metric_rows.extend(result["metric_rows"])
    walk_type_rows.extend(result["type_rows"])
    logger.info("ensemble_walk_fold_done fold=%s checkpoints=%s", fold_name, members)

walk_forward_threshold_summary = pd.DataFrame(walk_threshold_rows).set_index(["stage", "scope"])
walk_forward_evaluation_metrics = pd.DataFrame(walk_metric_rows).set_index(["stage", "strategy"])
walk_forward_type_evaluation = pd.DataFrame(walk_type_rows).set_index(["stage", "inspection_type"])
walk_forward_strategy_summary = (
    walk_forward_evaluation_metrics.reset_index().groupby("strategy").agg(
        folds=("stage", "nunique"), mean_pr_auc=("pr_auc", "mean"),
        mean_recall=("recall", "mean"), min_recall=("recall", "min"),
        recall_99_folds=("recall", lambda values: int((values >= MIN_RECALL).sum())),
        mean_false_call_reduction=("false_call_reduction", "mean"),
        min_false_call_reduction=("false_call_reduction", "min"),
        total_tp=("tp", "sum"), total_fn=("fn", "sum"),
    )
)
display(walk_forward_threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_evaluation_metrics[["positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(walk_forward_type_evaluation[["threshold", "positive_samples", "pr_auc", "recall", "false_call_reduction", "tp", "fn"]])
display(walk_forward_strategy_summary)
logger.info("walk_forward_strategy_summary=%s", walk_forward_strategy_summary.to_dict(orient="index"))

2026-08-25 13:58:10,082 | INFO | ensemble_walk_fold_done fold=fold_1 checkpoints=[0.3]


2026-08-25 13:58:10,349 | INFO | ensemble_walk_fold_done fold=fold_2 checkpoints=[0.3, 0.4]


2026-08-25 13:58:10,599 | INFO | ensemble_walk_fold_done fold=fold_3 checkpoints=[0.3, 0.4, 0.5]


threshold  positive_samples    recall  false_call_reduction  \
stage  scope                                                                 
fold_1 global   0.000018               200  0.990000              0.013203   
       type_0   0.002591                11  1.000000              0.399905   
       type_1   0.000700                20  1.000000              0.223753   
       type_2   0.000009                92  1.000000              0.003913   
       type_3   0.000409                73  1.000000              0.421372   
       type_4   0.002373                 4  1.000000              0.000000   
fold_2 global   0.000468               326  0.990798              0.316764   
       type_0   0.001018                50  1.000000              0.466646   
       type_1   0.000468               186  0.994624              0.132401   
       type_2   0.000133                49  1.000000              0.195242   
       type_3   0.009010                39  1.000000              0.715989   
       type_4   0.002917                 2  1.000000              0.000000   
fold_3 global   0.000158               152  0.993421              0.199160   
       type_0   0.000259                14  1.000000              0.431613   
       type_1   0.000674                80  1.000000              0.332592   
       type_2   0.000107                32  1.000000              0.034245   
       type_3   0.000321                23  1.000000              0.340716   
       type_4   0.003242                 3  1.000000              0.000000   

                tp  fn  
stage  scope            
fold_1 global  198   2  
       type_0   11   0  
       type_1   20   0  
       type_2   92   0  
       type_3   73   0  
       type_4    4   0  
fold_2 global  323   3  
       type_0   50   0  
       type_1  185   1  
       type_2   49   0  
       type_3   39   0  
       type_4    2   0  
fold_3 global  151   1  
       type_0   14   0  
       type_1   80   0  
       type_2   32   0  
       type_3   23   0  
       type_4    3   0

positive_samples    pr_auc  precision  \
stage  strategy                                                          
fold_1 fixed_0.5                              326  0.151108   0.267544   
       global_threshold                       326  0.151108   0.007472   
       type_specific_thresholds               326  0.151108   0.008997   
fold_2 fixed_0.5                              152  0.032925   0.053097   
       global_threshold                       152  0.032925   0.005203   
       type_specific_thresholds               152  0.032925   0.006882   
fold_3 fixed_0.5                               39  0.035162   0.125000   
       global_threshold                        39  0.035162   0.001017   
       type_specific_thresholds                39  0.035162   0.001184   

                                   recall  false_call_reduction        f1  \
stage  strategy                                                             
fold_1 fixed_0.5                 0.187117              0.996180  0.220217   
       global_threshold          1.000000              0.009356  0.014833   
       type_specific_thresholds  0.987730              0.188681  0.017832   
fold_2 fixed_0.5                 0.039474              0.997570  0.045283   
       global_threshold          0.927632              0.387828  0.010349   
       type_specific_thresholds  0.822368              0.590371  0.013650   
fold_3 fixed_0.5                 0.051282              0.999680  0.072727   
       global_threshold          0.974359              0.147807  0.002031   
       type_specific_thresholds  1.000000              0.249030  0.002365   

                                  tp   fn     fp     tn  
stage  strategy                                          
fold_1 fixed_0.5                  61  265    167  43547  
       global_threshold          326    0  43305    409  
       type_specific_thresholds  322    4  35466   8248  
fold_2 fixed_0.5                   6  146    107  43928  
       global_threshold          141   11  26957  17078  
       type_specific_thresholds  125   27  18038  25997  
fold_3 fixed_0.5                   2   37     14  43800  
       global_threshold           38    1  37338   6476  
       type_specific_thresholds   39    0  32903  10911

threshold  positive_samples    pr_auc    recall  \
stage  inspection_type                                                    
fold_1 0                 0.002591                50  0.099411  0.960000   
       1                 0.000700               186  0.234462  0.989247   
       2                 0.000009                49  0.330674  1.000000   
       3                 0.000409                39  0.570201  1.000000   
       4                 0.002373                 2  0.006154  1.000000   
fold_2 0                 0.001018                14  0.003358  0.571429   
       1                 0.000468                80  0.223545  1.000000   
       2                 0.000133                32  0.035561  0.968750   
       3                 0.009010                23  0.002102  0.130435   
       4                 0.002917                 3  0.004298  1.000000   
fold_3 0                 0.000259                 4  0.006735  1.000000   
       1                 0.000674                25  0.028674  1.000000   
       2                 0.000107                 7  0.048399  1.000000   
       3                 0.000321                 3  0.169440  1.000000   
       4                 0.003242                 0       NaN       NaN   

                        false_call_reduction   tp  fn  
stage  inspection_type                                 
fold_1 0                            0.678715   48   2  
       1                            0.181743  184   2  
       2                            0.001745   49   0  
       3                            0.217848   39   0  
       4                            0.000000    2   0  
fold_2 0                            0.726452    8   6  
       1                            0.391058   80   0  
       2                            0.045966   31   1  
       3                            0.827398    3  20  
       4                            0.000000    3   0  
fold_3 0                            0.223416    4   0  
       1                            0.119966   25   0  
       2                            0.050110    7   0  
       3                            0.548066    3   0  
       4                            0.000000    0   0

,folds,mean_pr_auc,mean_recall,min_recall,recall_99_folds,mean_false_call_reduction,min_false_call_reduction,total_tp,total_fn
strategy,,,,,,,,,
fixed_0.5,3,0.073065,0.092624,0.039474,0,0.997810,0.996180,69,448
global_threshold,3,0.073065,0.967330,0.927632,1,0.181664,0.009356,505,12
type_specific_thresholds,3,0.073065,0.936699,0.822368,1,0.342694,0.188681,486,31


2026-08-25 13:58:10,619 | INFO | walk_forward_strategy_summary={'fixed_0.5': {'folds': 3, 'mean_pr_auc': 0.07306483419465344, 'mean_recall': 0.0926240999699185, 'min_recall': 0.039473684210526314, 'recall_99_folds': 0, 'mean_false_call_reduction': 0.9978100985683014, 'min_false_call_reduction': 0.9961797135928993, 'total_tp': 69, 'total_fn': 448}, 'global_threshold': {'folds': 3, 'mean_pr_auc': 0.07306483419465344, 'mean_recall': 0.9673301844354475, 'min_recall': 0.9276315789473685, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.18166359054976552, 'min_false_call_reduction': 0.009356270302420278, 'total_tp': 505, 'total_fn': 12}, 'type_specific_thresholds': {'folds': 3, 'mean_pr_auc': 0.07306483419465344, 'mean_recall': 0.9366994941341082, 'min_recall': 0.8223684210526315, 'recall_99_folds': 1, 'mean_false_call_reduction': 0.34269408591514333, 'min_false_call_reduction': 0.18868097177105733, 'total_tp': 486, 'total_fn': 31}}


## 9. 최종 Validation 임계값 선택과 Walk-forward 모델의 Test 추론

In [9]:
validation_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_validation")
test_probability = mean_checkpoint_probability(FINAL_MEMBER_CHECKPOINTS, "final_test")
model_summary = pd.Series({"ensemble_members": len(FINAL_MEMBER_CHECKPOINTS), "member_checkpoints": FINAL_MEMBER_CHECKPOINTS}, name="final_ensemble")

final_result = evaluate_calibration_and_future(validation_df, validation_probability, test_df, test_probability, "final_test")
global_threshold_selection = final_result["global_selection"]
thresholds_by_type = final_result["type_thresholds"]
threshold_summary = pd.DataFrame(final_result["threshold_rows"]).set_index(["stage", "scope"])
type_selected_test_metrics = pd.DataFrame(final_result["type_rows"]).set_index(["stage", "inspection_type"])
test_strategy_metrics = pd.DataFrame(final_result["metric_rows"]).set_index(["stage", "strategy"])
validation_fixed_metrics = pd.Series(evaluate_probabilities(validation_df[TARGET], validation_probability), name="validation_fixed_0.5")
type_validation_rows, type_test_rows = [], []
for inspection_type in inspection_types:
    type_validation = validation_df.loc[validation_df[TYPE_COLUMN] == inspection_type]
    type_test = test_df.loc[test_df[TYPE_COLUMN] == inspection_type]
    type_validation_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_validation[TARGET], validation_probability.loc[type_validation.index])})
    type_test_rows.append({"inspection_type": inspection_type, **evaluate_probabilities(type_test[TARGET], test_probability.loc[type_test.index])})
type_validation_metrics = pd.DataFrame(type_validation_rows).set_index("inspection_type")
type_test_metrics = pd.DataFrame(type_test_rows).set_index("inspection_type")
display(model_summary)
display(threshold_summary[["threshold", "positive_samples", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(test_strategy_metrics[["pr_auc", "precision", "recall", "false_call_reduction", "f1", "tp", "fn", "fp", "tn"]])
display(type_selected_test_metrics[["threshold", "positive_samples", "pr_auc", "precision", "recall", "false_call_reduction", "tp", "fn", "fp", "tn"]])
display(type_validation_metrics)
display(type_test_metrics)
logger.info("validation_fixed_metrics=%s", validation_fixed_metrics.to_dict())
logger.info("test_strategy_metrics=%s", test_strategy_metrics.reset_index().to_dict(orient="records"))

ensemble_members                         4
member_checkpoints    [0.3, 0.4, 0.5, 0.7]
Name: final_ensemble, dtype: object

threshold  positive_samples    recall  \
stage      scope                                           
final_test global   0.000473               357  0.991597   
           type_0   0.000432                12  1.000000   
           type_1   0.001030               224  0.991071   
           type_2   0.000183                27  1.000000   
           type_3   0.001931                21  1.000000   
           type_4   0.003301                73  1.000000   

                   false_call_reduction   tp  fn     fp     tn  
stage      scope                                                
final_test global              0.534040  354   3  20348  23321  
           type_0              0.532123   12   0   6212   7065  
           type_1              0.494030  222   2   3136   3062  
           type_2              0.415195   27   0   4172   2962  
           type_3              0.832358   21   0   2721  13510  
           type_4              0.000000   73   0    829      0

pr_auc  precision    recall  \
stage      strategy                                                  
final_test fixed_0.5                 0.390438   0.715686  0.188387   
           global_threshold          0.390438   0.043008  0.939355   
           type_specific_thresholds  0.390438   0.048837  0.924731   

                                     false_call_reduction        f1    tp  \
stage      strategy                                                         
final_test fixed_0.5                             0.997970  0.298264   438   
           global_threshold                      0.433119  0.082251  2184   
           type_specific_thresholds              0.511542  0.092774  2150   

                                       fn     fp     tn  
stage      strategy                                      
final_test fixed_0.5                 1887    174  85553  
           global_threshold           141  48597  37130  
           type_specific_thresholds   175  41874  43853

threshold  positive_samples    pr_auc  precision  \
stage      inspection_type                                                     
final_test 0                 0.000432               195  0.071200   0.014979   
           1                 0.001030               774  0.545909   0.104490   
           2                 0.000183               731  0.563465   0.041694   
           3                 0.001931               612  0.258803   0.058003   
           4                 0.003301                13  0.017857   0.017857   

                              recall  false_call_reduction   tp  fn     fp  \
stage      inspection_type                                                   
final_test 0                0.784615              0.478597  153  42  10061   
           1                0.983204              0.436642  761  13   6522   
           2                0.956224              0.189077  699  32  16066   
           3                0.856209              0.752090  524  88   8510   
           4                1.000000              0.000000   13   0    715   

                               tn  
stage      inspection_type         
final_test 0                 9235  
           1                 5055  
           2                 3746  
           3                25817  
           4                    0

,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,13289,12,13277,0,12,0,0.999097,0.000000,0.000000,1.000000,0.000000,0.849100,0.004113
1,6422,224,6195,3,172,52,0.972750,0.945455,0.232143,0.999516,0.372760,0.965428,0.732704
2,7161,27,7132,2,18,9,0.997207,0.818182,0.333333,0.999720,0.473684,0.883822,0.386381
3,16252,21,16198,33,11,10,0.997293,0.232558,0.476190,0.997967,0.312500,0.969174,0.185716
4,902,73,829,0,73,0,0.919069,0.000000,0.000000,1.000000,0.000000,0.500000,0.080931


,rows,positive_samples,tn,fp,fn,tp,accuracy,precision,recall,false_call_reduction,f1,roc_auc,pr_auc
inspection_type,,,,,,,,,,,,,
0,19491,195,19296,0,195,0,0.989995,0.000000,0.000000,1.000000,0.000000,0.710260,0.071200
1,12351,774,11505,72,494,280,0.954174,0.795455,0.361757,0.993781,0.497336,0.904984,0.545909
2,20543,731,19804,8,659,72,0.967532,0.900000,0.098495,0.999596,0.177559,0.894075,0.563465
3,34939,612,34233,94,526,86,0.982255,0.477778,0.140523,0.997262,0.217172,0.872139,0.258803
4,728,13,715,0,13,0,0.982143,0.000000,0.000000,1.000000,0.000000,0.500000,0.017857


2026-08-25 13:58:11,257 | INFO | validation_fixed_metrics={'rows': 44026.0, 'positive_samples': 357.0, 'tn': 43631.0, 'fp': 38.0, 'fn': 286.0, 'tp': 71.0, 'accuracy': 0.9926407123063644, 'precision': 0.6513761467889908, 'recall': 0.19887955182072828, 'false_call_reduction': 0.9991298174906684, 'f1': 0.30472103004291845, 'roc_auc': 0.9385710866819419, 'pr_auc': 0.37415754379033184}


2026-08-25 13:58:11,259 | INFO | test_strategy_metrics=[{'stage': 'final_test', 'strategy': 'fixed_0.5', 'rows': 88052, 'positive_samples': 2325, 'tn': 85553, 'fp': 174, 'fn': 1887, 'tp': 438, 'accuracy': 0.9765933766410757, 'precision': 0.7156862745098039, 'recall': 0.18838709677419355, 'false_call_reduction': 0.9979703010720077, 'f1': 0.2982635342185904, 'roc_auc': 0.883872756365512, 'pr_auc': 0.39043769210795637}, {'stage': 'final_test', 'strategy': 'global_threshold', 'rows': 88052, 'positive_samples': 2325, 'tn': 37130, 'fp': 48597, 'fn': 141, 'tp': 2184, 'accuracy': 0.44648616726479806, 'precision': 0.04300821173273468, 'recall': 0.9393548387096774, 'false_call_reduction': 0.43311908733537857, 'f1': 0.08225059315331601, 'roc_auc': 0.883872756365512, 'pr_auc': 0.39043769210795637}, {'stage': 'final_test', 'strategy': 'type_specific_thresholds', 'rows': 88052, 'positive_samples': 2325, 'tn': 43853, 'fp': 41874, 'fn': 175, 'tp': 2150, 'accuracy': 0.5224526416208604, 'precision': 0.0

## 10. 원본 무결성과 종료 확인

In [10]:
DATA_SHA256_AFTER = sha256_file(DATA_PATH)
MAPPING_SHA256_AFTER = sha256_file(MAPPING_PATH)
assert DATA_SHA256_AFTER == DATA_SHA256_BEFORE
assert MAPPING_SHA256_AFTER == MAPPING_SHA256_BEFORE
verification = pd.Series({
    "dataset_sha256_unchanged": True, "mapping_sha256_unchanged": True,
    "trained_model_units": len(FINAL_MEMBER_CHECKPOINTS) * len(inspection_types), "final_ensemble_members": len(FINAL_MEMBER_CHECKPOINTS),
    "test_evaluated_with_walk_forward_model": True,
    "fixed_threshold": DECISION_THRESHOLD,
    "global_threshold": global_threshold_selection["threshold"],
    "type_thresholds": thresholds_by_type,
    "time_weighting": f"linear_{TIME_WEIGHT_MIN:.1f}_to_{TIME_WEIGHT_MAX:.1f}_by_checkpoint_type_train_time",
    "time_weight_degenerate_any": bool(ensemble_training_summary["time_weight_degenerate"].any()),
    "log_file": f"docs/peace/{LOG_PATH.name}",
}, name="verification")
display(verification)
logger.info("source_integrity=PASS walk_forward_test_model=True time_weighting=linear_by_checkpoint_type_train")
logger.info("experiment_complete=%s", EXPERIMENT_ID)
for handler in logger.handlers:
    handler.flush()

dataset_sha256_unchanged                                                               True
mapping_sha256_unchanged                                                               True
trained_model_units                                                                      20
final_ensemble_members                                                                    4
test_evaluated_with_walk_forward_model                                                 True
fixed_threshold                                                                         0.5
global_threshold                                                                   0.000473
type_thresholds                           {0: 0.0004324220208218321, 1: 0.00103001031675...
time_weighting                              linear_1.0_to_2.0_by_checkpoint_type_train_time
time_weight_degenerate_any                                                            False
log_file                                  docs/peace/0825_peace_011_type_expert_

2026-08-25 13:58:11,442 | INFO | source_integrity=PASS walk_forward_test_model=True time_weighting=linear_by_checkpoint_type_train


2026-08-25 13:58:11,442 | INFO | experiment_complete=0825_peace_011_type_expert_fold_ensemble_time_weight


## 11. 결론과 해석

- 모든 체크포인트×타입 학습 20건에서 `sample_weight`가 **1.0→2.0**으로 생성됐고, `time_weight_degenerate=False`였습니다.
- 최종 Test **PR-AUC는 0.390438**로, `005` Fold 앙상블의 **0.382545**와 `008` 단일 시간 가중치의 **0.371587**보다 높았습니다.
- Validation 공통 임계값 **0.000473**을 Test에 적용하면 Recall **93.94%**, False Call Reduction **43.31%**였습니다. Recall은 `005`와 같았지만 FCR은 `005`의 **52.04%**보다 낮았습니다.
- 타입별 임계값을 Test에 적용하면 Recall **92.47%**, False Call Reduction **51.15%**였습니다. 이는 `005` 타입별 전략의 **93.29% / 47.67%**보다 Recall은 낮고 FCR은 높았습니다.
- Walk-forward 공통 임계값의 평균 Recall은 **96.73%**, 최저 Recall은 **92.76%**로 `005`의 **98.68% / 96.05%**보다 미래 안정성이 약해졌습니다.
- 따라서 시간 가중치를 Fold 앙상블에 추가하면 ranking(PR-AUC)은 좋아졌지만, Walk-forward 안정성과 공통 임계값 FCR이 악화됐습니다. 최악 미래 Recall과 공통 임계값 운영 효율을 우선하면 **내부 Champion은 `005` Fold 앙상블로 유지**합니다.
